## (1) Importing modules

In [ ]:

#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-05-10
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

import numpy as np
import pandas as pd

from obspy import UTCDateTime

import torch
import torch.nn as nn
print("PyTorch version:", torch.__version__) # PyTorch version: 1.12.1

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# region <add the sys.path to search for custom modules>
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent.parent
sys.path.append(str(project_root))


print(f"Current dir: {current_dir}\n"
      f"Project root: {project_root}")
# endregion


# import the custom functions

## (1) LSTM model

### Do not worry, if you can not figure out the model structure

In [ ]:
class LSTM_Attention(nn.Module):
    
    def __init__(self, feature_size, device,
                 hidden_size=256, num_layers=4,
                 dropout=0.25, bidirectional=False, output_dim=2):
        super().__init__()

        # Keep original structure
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.device = device

        # same as "D = 2 if bidirectional=True otherwise 1" in Pytorch
        if bidirectional is True:
            self.D = 2
        else:
            self.D = 1

        # lstm layer
        self.lstm = nn.LSTM(
            input_size=feature_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bias=True,
            batch_first=True,
            dropout=dropout,
            bidirectional=bidirectional)

        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # set forget gate bias to 1 to help memory retention
                n = param.size(0)
                param.data[n // 4:n // 2].fill_(1.0)

        # attention layer
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * self.D,
            num_heads=min(8, num_layers),
            dropout=dropout,
            batch_first=True)

        # layer normalization
        self.layer_norm = nn.LayerNorm(hidden_size * self.D)

        # learnable attention pooling parameters
        self.attn_pool_weight = nn.Parameter(torch.Tensor(hidden_size * self.D, 1))
        nn.init.xavier_uniform_(self.attn_pool_weight)

        # output layer
        self.fully_connect = nn.Sequential(
            nn.Linear(hidden_size * self.D, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_dim)
        )
        
        nn.init.xavier_uniform_(self.fully_connect[0].weight)
        nn.init.xavier_uniform_(self.fully_connect[-1].weight)



    def forward(self, x, t=None, hidden=None):
        x = x.to(torch.float32)
        
        print(f"Enter the LSTM_Attention model")
        print(f"x.shape: {x.shape}")
        
        self.lstm.flatten_parameters()

        # Initialize hidden states (keep your original logic)
        if hidden is None:
            h0 = torch.zeros(self.num_layers * self.D, x.size(0), self.hidden_size, dtype=x.dtype).to(self.device)
            c0 = torch.zeros(self.num_layers * self.D, x.size(0), self.hidden_size, dtype=x.dtype).to(self.device)
        else:
            h0, c0 = hidden
            h0, c0 = h0.to(self.device), c0.to(self.device)

        print(f"h0.shape: {h0.shape}")
        print(f"c0.shape: {c0.shape}")
        
        # output.shape = ([batch_size, sequence_length, hidden_size])
        output, (h_n, c_n) = self.lstm(x, (h0, c0))
        # layer normalization
        output = self.layer_norm(output)
        print(f"output.shape: {output.shape}")
        
        # attention with residual connection
        attn_output, _ = self.attention(output, output, output)
        print(f"attn_output.shape: {attn_output.shape}")
        
        output = output + attn_output  # skip connection helps gradient flow

        # pooling and output
        # pooled_output = torch.mean(output, dim=1)
        attn_scores = torch.matmul(output, self.attn_pool_weight)  # Shape: (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)  # normalize scores
        pooled_output = torch.sum(output * attn_weights, dim=1)  # weighted sum across time steps

        output_final = self.fully_connect(pooled_output)
        print(f"output_final.shape: {output_final.shape}")
        print(f"Leave the LSTM_Attention model\n")
        

        return output_final

## (2) Load the data and see what we will get

### (2-1) Prepare the paramsters

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# note: now we are runing the "H" features -> feature_size=12
model = LSTM_Attention(feature_size=12, device=device, dropout=0.25)
model.to(device)

class_weight_binary = torch.tensor([1 - 0.9, 0.9]).to(device) # 0.9 for event
loss_func = torch.nn.CrossEntropyLoss(reduction="mean", weight=class_weight_binary, label_smoothing=0.05)

In [ ]:
dataloader = torch.load(f"{current_dir}/data/2017_dataloader.pt")

### (2-2) Model training

In [ ]:
for batch_data in dataloader.dataLoader():
    # t_features of t to t_{sequence_length}, shape ([batch_size, sequence_length]), float time stamps
    t_features = batch_data['t_features']

    # features of t to t_i, shape ([batch_size, sequence_length, num_features])
    features = batch_data['features']

    # t_target of t_{sequence_length+1}, shape ([batch_size, 1]), float time stamps
    t_target = batch_data['t_target']

    # target of t_{sequence_length+1}, shape ([batch_size, 1]), debris flow probability or label
    target = batch_data['target']



    # pass the seismic features and return the raw logits
    raw_logits = model(features, t_features)  # return the model output logits, shape (batch_size, 2)

    # convert the raw_logits to event probability
    predicted_pro = torch.softmax(raw_logits, dim=1)[:, 1] # for my case, the second column is event probability

    # calculate the current losss
    loss = loss_func(raw_logits, target)
    

    # # check the shape changes for input and output
    print(features.shape)
    print(raw_logits.shape)
    print(predicted_pro.shape)
    print(f"predicted_pro[0]: {predicted_pro[0]}")
    break

## Question 1: How to let the model know the "loss"?